In [2]:
!pip install pandas --quiet

In [3]:
# importando as bibliotecas
import pandas as pd
import matplotlib.pyplot as plt

In [4]:
df = pd.read_csv('Ibovespa_ml.csv')
df.head()

,Data,Último,Abertura,Máxima,Mínima,Vol.,Var%
0,25.03.2025,132.068,131.327,133.471,131.325,"9,24B","0,57%"
1,24.03.2025,131.321,132.344,132.424,130.992,"8,30B","-0,77%"
2,21.03.2025,132.345,132.005,132.588,131.776,"14,19B","0,30%"
3,20.03.2025,131.955,132.505,132.713,131.813,"14,19B","-0,42%"
4,19.03.2025,132.508,131.476,132.984,131.451,"12,20B","0,79%"


| Coluna | Descrição  | Insights Chave |
| :--- | :--- | :--- |
| **Data**  | **Referência Temporal:** O dia de fechamento do pregão. É a base para ordenar e analisar a série temporal.  | Usada como índice temporal. |
| **Último**  | **Preço de Fechamento:** O valor final do índice no encerramento das negociações.  | **Fundamental** para o cálculo de *features* e da variável alvo. |
| **Abertura**  | **Preço de Abertura:** O valor do índice no início do pregão.  | Ajuda a medir o *gap* de preço em relação ao dia anterior. |
| **Máxima**  | **Preço Máximo:** O maior valor atingido pelo índice durante o pregão. | Indica a força da pressão de compra no dia. |
| **Mínima**  | **Preço Mínimo:** O menor valor atingido pelo índice durante o pregão.  | Indica a força da pressão de venda no dia. |
| **Vol.**  | **Volume de Negócios:** O valor total transacionado naquele dia.  | Reflete a liquidez e a convicção por trás dos movimentos de preço. |
| **Var%**  | **Variação Percentual Diária:** A diferença percentual entre o Fechamento de hoje e o Fechamento de ontem.  | Indica o resultado bruto do dia. **Não deve ser usada como *feature* preditiva.** |

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Data      5000 non-null   object 
 1   Último    5000 non-null   float64
 2   Abertura  5000 non-null   float64
 3   Máxima    5000 non-null   float64
 4   Mínima    5000 non-null   float64
 5   Vol.      4999 non-null   object 
 6   Var%      5000 non-null   object 
dtypes: float64(4), object(3)
memory usage: 273.6+ KB


In [6]:
# Converte a coluna 'DATA' para o tipo datetime
df['Data'] = pd.to_datetime(df['Data'])
# Ordenar o DataFrame
df.sort_values(by='Data', inplace=True)
# Definir 'Data' como o índice
df.set_index('Data', inplace=True)
df.head()

/tmp/ipython-input-1963718354.py:2: UserWarning: Parsing dates in %d.%m.%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df['Data'] = pd.to_datetime(df['Data'])


,Último,Abertura,Máxima,Mínima,Vol.,Var%
Data,,,,,,
2005-01-17,24.515,24.924,25.022,24.515,"75,06M","-1,64%"
2005-01-18,24.089,24.515,24.515,24.019,"179,42M","-1,74%"
2005-01-19,24.271,24.091,24.465,24.091,"113,11M","0,76%"
2005-01-20,23.610,24.271,24.271,23.534,"182,06M","-2,72%"
2005-01-21,23.818,23.618,24.006,23.609,"97,40M","0,88%"


In [7]:
import pandas as pd

def convert_to_float(value):
    if isinstance(value, str):
        # Padroniza o formato numérico: remove '.' de milhar e troca ',' por '.' decimal
        value = value.replace('.', '').replace(',', '.')

        if 'B' in value:
            # Converte Bilhões (B)
            return float(value.replace('B', '')) * 1_000_000_000
        elif 'M' in value:
            # Converte Milhões (M)
            return float(value.replace('M', '')) * 1_000_000
        elif 'K' in value:
            # Converte Milhares (K)
            return float(value.replace('K', '')) * 1_000

        # AJUSTE: Converte o valor para float se não tiver sufixo
        return float(value)

    # Retorna o valor original se ele já for um número (int, float, etc.)
    return value

# Aplica a função na coluna 'Vol.'
df['Vol.'] = df['Vol.'].apply(convert_to_float)

# Mostra o resultado e as informações para confirmar o tipo de dado (dtype)
print(df.head())
print("\nInformações do DataFrame após a conversão:")
df.info()

            Último  Abertura  Máxima  Mínima         Vol.    Var%
Data                                                             
2005-01-17  24.515    24.924  25.022  24.515   75060000.0  -1,64%
2005-01-18  24.089    24.515  24.515  24.019  179420000.0  -1,74%
2005-01-19  24.271    24.091  24.465  24.091  113110000.0   0,76%
2005-01-20  23.610    24.271  24.271  23.534  182060000.0  -2,72%
2005-01-21  23.818    23.618  24.006  23.609   97400000.0   0,88%

Informações do DataFrame após a conversão:
<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 5000 entries, 2005-01-17 to 2025-03-25
Data columns (total 6 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Último    5000 non-null   float64
 1   Abertura  5000 non-null   float64
 2   Máxima    5000 non-null   float64
 3   Mínima    5000 non-null   float64
 4   Vol.      4999 non-null   float64
 5   Var%      5000 non-null   object 
dtypes: float64(5), object(1)
memory usage: 273.4+ KB


In [8]:
def convert_percentage_to_float(value):
    if isinstance(value, str) and '%' in value:
        value = value.replace('%', '').replace(',', '.')
        if '-' in value:
            value = value.replace('-', '')
            return float(value) * -1
        else:
            return float(value)
    return value
df['Var%'] = df['Var%'].apply(convert_percentage_to_float)

print(df.head())

            Último  Abertura  Máxima  Mínima         Vol.  Var%
Data                                                           
2005-01-17  24.515    24.924  25.022  24.515   75060000.0 -1.64
2005-01-18  24.089    24.515  24.515  24.019  179420000.0 -1.74
2005-01-19  24.271    24.091  24.465  24.091  113110000.0  0.76
2005-01-20  23.610    24.271  24.271  23.534  182060000.0 -2.72
2005-01-21  23.818    23.618  24.006  23.609   97400000.0  0.88


In [9]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
Último,5000.0,7.287309e+01,2.942822e+01,23.610,5.181575e+01,6.348900e+01,1.007542e+02,1.373440e+02
Abertura,5000.0,7.285223e+01,2.942406e+01,23.618,5.181275e+01,6.348500e+01,1.007375e+02,1.373490e+02
Máxima,5000.0,7.354927e+01,2.958255e+01,24.006,5.238525e+01,6.409950e+01,1.016625e+02,1.374690e+02
Mínima,5000.0,7.216412e+01,2.927232e+01,23.534,5.124275e+01,6.283500e+01,9.989200e+01,1.366640e+02
Vol.,4999.0,1.335862e+08,1.122452e+09,112100.000,2.830000e+06,4.480000e+06,1.107000e+07,2.487000e+10
Var%,5000.0,4.723800e-02,1.662822e+00,-14.780,-8.125000e-01,6.000000e-02,9.400000e-01,1.466000e+01


In [10]:
df.isnull().sum()

,0
Último,0
Abertura,0
Máxima,0
Mínima,0
Vol.,1
Var%,0


---
Esta seção concentra-se em preparar o DataFrame (`df`) para a modelagem, garantindo que todas as colunas estejam no formato correto (data e numérico `float`).

###  Tratamento e Indexação da Coluna 'Data'

| Ação | Descrição |
| :--- | :--- |
| **Conversão** | A coluna `'Data'` foi convertida do formato de texto para o tipo `datetime` do Pandas. |
| **Ordenação** | O DataFrame foi ordenado cronologicamente para garantir a ordem correta na análise de séries temporais. |
| **Indexação** | A coluna `'Data'` foi definida como o índice do DataFrame. |

###  Limpeza e Conversão de Colunas Numéricas

As colunas que continham valores numéricos como texto, separadores de milhar incorretos (ponto) e sufixos de unidade foram limpas.

| Coluna | Ação | Regex/Lógica |
| :--- | :--- | :--- |
| **`Ultimo`, `Abertura`, `Máxima`, `Mínima`** | As colunas de preço foram convertidas para `float` (números decimais). |  `replace('.', '').replace(',', '.')` |
| **`Vol.`** | A coluna de volume foi convertida para `float`, tratando os sufixos de unidade. |  Converte 'B', 'M', 'K' para bilhões, milhões e milhares, respectivamente. |
| **`Var%`** | A coluna de variação percentual foi convertida para `float`, removendo o `%`. |  Remove `%`, troca `,` por `.`, e trata o sinal negativo (`-`). |

---

In [11]:
df

,Último,Abertura,Máxima,Mínima,Vol.,Var%
Data,,,,,,
2005-01-17,24.515,24.924,25.022,24.515,7.506000e+07,-1.64
2005-01-18,24.089,24.515,24.515,24.019,1.794200e+08,-1.74
2005-01-19,24.271,24.091,24.465,24.091,1.131100e+08,0.76
2005-01-20,23.610,24.271,24.271,23.534,1.820600e+08,-2.72
2005-01-21,23.818,23.618,24.006,23.609,9.740000e+07,0.88
...,...,...,...,...,...,...
2025-03-19,132.508,131.476,132.984,131.451,1.220000e+10,0.79
2025-03-20,131.955,132.505,132.713,131.813,1.419000e+10,-0.42
2025-03-21,132.345,132.005,132.588,131.776,1.419000e+10,0.30


In [12]:
# Salva o DataFrame em um arquivo chamado 'meu_arquivo.csv'
df.to_csv('Fase_01.csv', index=True)